# Testing an improved screening approach

First, user enters their question.

Then platform asks:

1) Are you interested in a particular population? (provides three examples from more specific to less specific + option "keep it broad" + free text option that the user can add, and option to add more)

2) Are you interested in a particular outcome? (similar set of examples as above)

3) Set search parameters: source, geography and time window

4) Any other specific inclusion criteria? (could even skip this for now)

5) Any other specific exclusion criteria? (could even skip this for now)

Importantly we do not add the inputs from 1-5 to any boolean query explicitly. Instead, we store this in a json object that we send to the database.
And we'll use it do generate the boolean and semantic queries in the subsequent step of retrieval, as well as for the screening step.

Provide a subtitle for each page, that says "We will use these to filter only the most relevant documents"

6) Summary window  summarising the search so far, also formulating the implied research question so far, eg "What interventions could [achieve outcome] on a [specified population], or something else that's tailored to the user input. Then the UI also prompts the user, if they have other questions they are interested in, such as (two examples and a free text field with options to add more, eg "what supporting factors are important for these interventions work?)

We then generate the boolean queries for OpenAlex (if querying that dataset required).

We also generate a semantic search query for Overton, which takes into account the positive informaton (user query, population, outcomes, research question, inclusion criteria)

In [1]:
import asyncio
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from app.services.analysis.references import ReferencesService
from app.services.analysis.relevance import RelevanceService

In [2]:
# Example query similar to the prompting notebook
QUERY = "heat pumps"
EXPORT_DIR = Path("./temp/screening_demo")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

async def run_retrieval_and_relevance():
    references_service = ReferencesService(export_dir=str(EXPORT_DIR))

    references_csv, boolean_queries, semantic_query, updated_context = await references_service.build_references(
        query=QUERY,
        sources=["openalex"],
        limit=50,
        mode="boolean",  # triggers boolean + enrichment + sampling
        search_context=None,
        n_query_runs=1,
    )

    print("References CSV:", references_csv)
    print("Boolean queries (all variants):", boolean_queries)
    print("Semantic query (Overton):", semantic_query)
    if updated_context and updated_context.get("comprehensiveness_estimate"):
        print("Comprehensiveness estimate:", updated_context.get("comprehensiveness_estimate"))

    # Run relevance screening only (stop before acquisition/parsing/extraction)
    relevance_service = RelevanceService(
        query=QUERY,
        export_dir=str(EXPORT_DIR),
        search_context=updated_context,
        relevance_output_path=str(EXPORT_DIR / "relevance_cache.jsonl"),
        keep_relevance_output=True,
    )
    screened_csv = await relevance_service.check_relevance(str(references_csv))
    print("Screened CSV:", screened_csv)

    df = pd.read_csv(screened_csv)
    print("Rows:", len(df))
    display(df.head(10))

await run_retrieval_and_relevance()


References CSV: temp/screening_demo/references.csv
Boolean queries (all variants): ['("heat pump" OR "heat pumps" OR "heat-pump" OR "heat-pumps" OR "air source heat pump" OR "air-source heat pump" OR "air source heat pumps" OR "air-source heat pumps" OR "ground source heat pump" OR "ground-source heat pump" OR "ground source heat pumps" OR "ground-source heat pumps" OR "geothermal heat pump" OR "geothermal heat pumps")', '(("heat pump" OR "heat pumps" OR "heat-pump" OR "heat-pumps" OR "air source heat pump" OR "air-source heat pump" OR "air source heat pumps" OR "air-source heat pumps" OR "ground source heat pump" OR "ground-source heat pump" OR "ground source heat pumps" OR "ground-source heat pumps" OR "geothermal heat pump" OR "geothermal heat pumps")) AND ("systematic review" OR "meta-analysis" OR "narrative synthesis")', '(("heat pump" OR "heat pumps" OR "heat-pump" OR "heat-pumps" OR "air source heat pump" OR "air-source heat pump" OR "air source heat pumps" OR "air-source heat p

,doc_id,source,source_id,title,abstract_or_summary,year,doi,authors,landing_page_url,pdf_url,...,source_country,relevance_score,retrieval_query_type,retrieval_priority,is_relevant,relevance_confidence,relevance_reason,top_line,document_type,document_type_reason
0,https://doi.org/10.1016/j.renene.2019.08.096,openalex,https://openalex.org/W2969253469,A systematic review of recent air source heat ...,No abstract available,2019,https://doi.org/10.1016/j.renene.2019.08.096,"['Xinru Wang', 'Liang Xia', 'Chris Bales', 'Xi...",http://arxiv.org/abs/2102.09797,https://arxiv.org/pdf/2102.09797,...,"China, Sweden",1887.97460,systematic_review,0,True,0.8,The document discusses air source heat pump sy...,The review focuses on air source heat pumps an...,reviews,This document is classified as a review becaus...
1,https://doi.org/10.1016/j.rser.2021.111830,openalex,https://openalex.org/W3214356147,A systematic review on optimal analysis of hor...,No abstract available,2021,https://doi.org/10.1016/j.rser.2021.111830,"['Gaoyang Hou', 'Hessam Taherian', 'Ying Song'...",https://doi.org/10.1016/j.rser.2021.111830,NaN,...,"China, United States",1557.26350,systematic_review,0,True,0.8,The document discusses horizontal heat exchang...,The review analyzes horizontal heat exchangers...,reviews,This document is classified as a review becaus...
2,https://doi.org/10.1016/j.rser.2015.10.157,openalex,https://openalex.org/W2176956381,Solar assisted heat pump systems for low tempe...,No abstract available,2015,https://doi.org/10.1016/j.rser.2015.10.157,"['Mahmut Sami Büker', 'Saffa Riffat']",https://doi.org/10.1016/j.rser.2015.10.157,NaN,...,United Kingdom,905.12683,systematic_review,0,True,0.8,The document discusses solar-assisted heat pum...,The review focuses on solar-assisted heat pump...,reviews,This document is classified as a review becaus...
3,https://doi.org/10.1016/j.est.2024.114097,openalex,https://openalex.org/W4403526342,Energy storage-integrated ground-source heat p...,© 2024 The Authors. Published by Elsevier Ltd....,2024,https://doi.org/10.1016/j.est.2024.114097,"['Arslan Saleem', 'Tehmina Ambreen', 'Carlos E...",https://doi.org/10.1016/j.est.2024.114097,https://doi.org/10.1016/j.est.2024.114097,...,NaN,640.71510,systematic_review,0,True,1.0,The document directly addresses heat pumps by ...,The review examines energy storage-integrated ...,reviews,This document is classified as a review becaus...
4,https://doi.org/10.1016/j.enbuild.2013.07.064,openalex,https://openalex.org/W2016493056,Meta-analysis of European heat pump field tria...,No abstract available,2013,https://doi.org/10.1016/j.enbuild.2013.07.064,"['Colin Gleeson', 'Robert Lowe']",https://doi.org/10.1016/j.enbuild.2013.07.064,NaN,...,United Kingdom,604.77704,systematic_review,0,True,0.8,The document focuses on heat pump efficiencies...,European heat pump field trials show varying e...,reviews,This document is classified as a review becaus...
5,https://doi.org/10.1016/j.desal.2022.116081,openalex,https://openalex.org/W4295238064,Recent advances in heat pump-coupled desalinat...,No abstract available,2022,https://doi.org/10.1016/j.desal.2022.116081,"['Huan Liu', 'Abanob Joseph', 'Mamoun M. Elsay...",https://doi.org/10.1016/j.desal.2022.116081,NaN,...,"China, Egypt",483.88974,systematic_review,0,True,0.8,The document discusses heat pump-coupled desal...,Heat pump-coupled desalination systems represe...,reviews,This document is classified as a review becaus...
6,https://doi.org/10.1016/j.solener.2023.112299,openalex,https://openalex.org/W4391517759,A systematic review of photovoltaic/thermal ap...,No abstract available,2024,https://doi.org/10.1016/j.solener.2023.112299,"['Hussein A. Kazem', 'Miqdam T. Chaichan', 'Al...",https://doi.org/10.1016/j.solener.2023.112299,NaN,...,"Iraq, Malaysia, Oman",412.14300,systematic_review,0,True,0.8,The document discusses photovoltaic/thermal ap...,The review examines photovoltaic/thermal appli...,reviews,This document is classified as a review becaus...

In [4]:
len(df)

NameError: name 'df' is not defined